In [11]:
# 1. Clonar repositorio e instalar dependencias

import os
import shutil

REPO_URL = "https://github.com/dibujoudea-boop/analizador_localizacion_login.git"
REPO_DIR = "analizador_localizacion_login"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL}
%cd {REPO_DIR}

!pip install -q -r requirements.txt

Cloning into 'analizador_localizacion_login'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 222 (delta 51), reused 15 (delta 15), pack-reused 153 (from 1)
Receiving objects: 100% (222/222), 3.25 MiB | 16.32 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/analizador_localizacion_login/analizador_localizacion_login


In [12]:
# 2. Instalcion de libreriras de pruebas
!pip install -q httpx
print("listo")

listo


In [13]:
%%writefile src/capture.py
# 3. Creación del módulo de captura

from __future__ import annotations
from datetime import datetime, timezone
from fastapi import Request


def get_client_ip(request: Request) -> str:
    forwarded = request.headers.get("x-forwarded-for")
    if forwarded:
        return forwarded.split(",")[0].strip()
    return request.client.host if request.client else "0.0.0.0"


def capture_signals(request: Request) -> dict:
    return {
        "ip": get_client_ip(request),
        "user_agent": request.headers.get("user-agent", ""),
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }



Writing src/capture.py


In [14]:
%%writefile src/enrichment.py
# 4. Creación del módulo de enriquecimiento

from __future__ import annotations

import json
import urllib.request
from ipaddress import ip_address

_NEUTRAL = {
    "declared_city": None,
    "declared_region": None,
    "declared_country": None,
    "lat": None,
    "lon": None,
    "timezone_offset": None,
    "asn": None,
    "geo_ok": False,
}


def _is_geolocatable(ip: str) -> bool:
    try:
        addr = ip_address(ip)
        return not (addr.is_private or addr.is_loopback or addr.is_reserved
                    or addr.is_link_local or addr.is_unspecified)
    except ValueError:
        return False


def geolocate_ip(ip: str, timeout: int = 5) -> dict:
    result = dict(_NEUTRAL)
    if not _is_geolocatable(ip):
        return result

    fields = "status,message,countryCode,regionName,city,lat,lon,timezone,as"
    url = f"http://ip-api.com/json/{ip}?fields={fields}"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        if data.get("status") == "success":
            result.update({
                "declared_city": data.get("city"),
                "declared_region": data.get("regionName"),
                "declared_country": data.get("countryCode"),
                "lat": data.get("lat"),
                "lon": data.get("lon"),
                "timezone_offset": data.get("timezone"),
                "asn": data.get("as"),
                "geo_ok": True,
            })
    except Exception:
        pass
    return result


def enrich_geo(event_dict: dict) -> dict:
    geo = geolocate_ip(event_dict.get("ip", ""))
    if geo["geo_ok"]:
        event_dict["declared_city"] = geo["declared_city"] or event_dict.get("declared_city")
        event_dict["declared_region"] = geo["declared_region"]
        event_dict["declared_country"] = geo["declared_country"] or event_dict.get("declared_country")
        event_dict["timezone_offset"] = geo["timezone_offset"]
        event_dict["asn"] = geo["asn"]
        event_dict["_lat"] = geo["lat"]
        event_dict["_lon"] = geo["lon"]
    return event_dict


Writing src/enrichment.py


In [15]:
%%writefile src/geo_cache.py
# 5. Creación de cache de geolocalizaciones

from __future__ import annotations

import json
from pathlib import Path

from src import enrichment

# Caché en disco de respuestas de geolocalización. Clave = IP, valor = dict
# devuelto por enrichment.geolocate_ip. Aporta dos cosas:
#   1) Reproducibilidad: una API online devuelve datos que dependen de la red
#      y del momento; con caché, re-ejecutar el notebook da el mismo resultado.
#   2) Ahorro de peticiones: evita agotar el límite gratuito de ip-api
#      (~45 peticiones/minuto) al repetir IPs.
CACHE_PATH = Path("data/geo_cache.json")

_installed = False


def _load() -> dict:
    if CACHE_PATH.exists():
        try:
            return json.loads(CACHE_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def _save(cache: dict) -> None:
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_text(
        json.dumps(cache, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def install_cache() -> None:
    """Envuelve ``enrichment.geolocate_ip`` con una caché en disco.

    No modifica ``enrichment.py``: sustituye la referencia del módulo por una
    versión cacheada. Como ``enrich_geo`` invoca ``geolocate_ip`` por su nombre
    global, empezará a usar la caché de forma transparente. Es idempotente.
    """
    global _installed
    if _installed:
        return

    original = enrichment.geolocate_ip

    def cached(ip: str, timeout: int = 5) -> dict:
        cache = _load()
        if ip in cache:
            return cache[ip]
        result = original(ip, timeout=timeout)
        # Solo se cachean respuestas válidas; los fallos se reintentan.
        if result.get("geo_ok"):
            cache[ip] = result
            _save(cache)
        return result

    enrichment.geolocate_ip = cached
    _installed = True

Writing src/geo_cache.py


In [16]:
%%writefile src/history.py
# 6. Creacion módulo para consultar el historico de las ultimos 10 eventos

from __future__ import annotations

import json
import sqlite3
from collections import Counter
from datetime import datetime
from math import radians, sin, cos, asin, sqrt
from pathlib import Path
from typing import Any, Dict, List, Optional

from src.storage import DB_PATH, init_db

# Número de eventos previos del usuario que se consideran para construir su
# perfil habitual (ubicación, zona horaria, dispositivos conocidos).
DEFAULT_WINDOW = 10

# Tope de velocidad (km/h) para evitar valores absurdos cuando el intervalo
# entre dos eventos es casi nulo (p.ej. accesos casi simultáneos desde dos
# países). Basta con superar holgadamente el umbral de "viaje imposible".
MAX_SPEED_KMH = 100000.0


def _haversine_km(lat1, lon1, lat2, lon2) -> float:
    """Distancia en kilómetros entre dos coordenadas (fórmula de haversine)."""
    r = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * r * asin(sqrt(a))


def _parse_iso(value: Any) -> Optional[datetime]:
    try:
        return datetime.fromisoformat(str(value))
    except (ValueError, TypeError):
        return None


def get_user_history(
    user_id: str,
    n: int = DEFAULT_WINDOW,
    db_path: Path = DB_PATH,
) -> List[Dict[str, Any]]:
    """Devuelve los N eventos previos del usuario (más reciente primero).

    Cada elemento es el evento crudo (``raw_event``) ya deserializado, que
    contiene las señales derivadas por el enriquecimiento (_lat, _lon,
    declared_city, timezone_offset, user_agent, etc.).
    """
    init_db(db_path)
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT raw_event
            FROM login_events
            WHERE user_id = ?
            ORDER BY id DESC
            LIMIT ?
            """,
            (user_id, n),
        ).fetchall()

    history: List[Dict[str, Any]] = []
    for row in rows:
        try:
            history.append(json.loads(row["raw_event"]))
        except (json.JSONDecodeError, TypeError):
            continue
    return history


def _mode(values: List[Any]) -> Optional[Any]:
    """Valor más frecuente de una lista, ignorando vacíos."""
    limpios = [v for v in values if v]
    if not limpios:
        return None
    return Counter(limpios).most_common(1)[0][0]


def derive_history_signals(
    event: Dict[str, Any],
    n: int = DEFAULT_WINDOW,
    db_path: Path = DB_PATH,
) -> Dict[str, Any]:
    """Deriva las señales de secuencia a partir del historial del usuario.

    Devuelve un diccionario con las señales que consume el motor de scoring.
    Para un usuario SIN historial se aplica una línea base neutra: no se
    penaliza el primer acceso (no hay comportamiento previo con el que
    comparar); el perfil de riesgo emerge a partir del segundo evento.
    """
    user_id = event.get("user_id", "")
    history = get_user_history(user_id, n=n, db_path=db_path)

    señales: Dict[str, Any] = {
        "known_location": None,
        "country_change": 0,
        "distance_from_prev_km": 0.0,
        "time_since_prev_min": 0.0,
        "travel_speed_kmh": 0.0,
        "is_new_device": 0,
        "timezone_mismatch": 0,
    }

    # Usuario nuevo: sin historial, línea base neutra.
    if not history:
        return señales

    # Ubicación y zona horaria habituales = moda de la ventana de eventos.
    señales["known_location"] = _mode([h.get("declared_city") for h in history])
    tz_habitual = _mode([h.get("timezone_offset") for h in history])

    prev = history[0]  # evento inmediatamente anterior (más reciente)

    # Cambio de país respecto al último acceso.
    pais_actual = event.get("declared_country")
    pais_prev = prev.get("declared_country")
    if pais_actual and pais_prev and pais_actual != pais_prev:
        señales["country_change"] = 1

    # Distancia, tiempo y velocidad respecto al último acceso.
    lat_a, lon_a = event.get("_lat"), event.get("_lon")
    lat_p, lon_p = prev.get("_lat"), prev.get("_lon")
    if None not in (lat_a, lon_a, lat_p, lon_p):
        dist = _haversine_km(lat_p, lon_p, lat_a, lon_a)
        señales["distance_from_prev_km"] = round(dist, 2)

        t_actual = _parse_iso(event.get("timestamp"))
        t_prev = _parse_iso(prev.get("timestamp"))
        if t_actual and t_prev:
            minutos = abs((t_actual - t_prev).total_seconds()) / 60.0
            señales["time_since_prev_min"] = round(minutos, 2)
            if minutos > 0:
                velocidad = dist / (minutos / 60.0)
            else:
                # Dos accesos en el mismo instante desde puntos distantes:
                # desplazamiento físicamente imposible.
                velocidad = MAX_SPEED_KMH if dist > 0 else 0.0
            señales["travel_speed_kmh"] = round(min(velocidad, MAX_SPEED_KMH), 2)

    # Dispositivo nuevo: el user-agent actual no aparece en el historial.
    ua_actual = (event.get("user_agent") or "").strip().lower()
    ua_previos = {(h.get("user_agent") or "").strip().lower() for h in history}
    if ua_actual and ua_actual not in ua_previos:
        señales["is_new_device"] = 1

    # Desajuste de zona horaria respecto a la habitual del usuario.
    tz_actual = event.get("timezone_offset")
    if tz_habitual and tz_actual and tz_actual != tz_habitual:
        señales["timezone_mismatch"] = 1

    return señales

Writing src/history.py


In [17]:
%%writefile src/pipeline.py
# 7. Creación del pipeline

from __future__ import annotations

from typing import Any, Dict

from fastapi import Request

from src.capture import capture_signals
from src import enrichment
from src import geo_cache
from src.history import derive_history_signals
from src.risk_engine import score_login_event
from src.storage import save_event_result
from src.schemas import RiskResponse

# Activa la caché sobre enrichment.geolocate_ip al importar el pipeline.
geo_cache.install_cache()


def run_raw_pipeline(body: Dict[str, Any], request: Request) -> RiskResponse:
    """Flujo funcional end-to-end para un evento de login crudo.

    Recibe únicamente lo aportado por el cliente (user_id y, opcionalmente,
    known_location), captura las señales reales de la petición HTTP, enriquece
    con geolocalización y reutiliza el motor y la persistencia de la v4 SIN
    modificarlos.
    """
    # 1) Captura de señales reales de la petición (IP, user-agent, hora).
    captured = capture_signals(request)

    # 2) Evento base: identidad del cliente + señales capturadas.
    event: Dict[str, Any] = {
        "user_id": body.get("user_id"),
        "ip": captured["ip"],
        "user_agent": captured["user_agent"],
        "timestamp": captured["timestamp"],
        "network_type": "residential",
        "scenario_type": "api_event_raw",
    }

    # 3) Enriquecimiento geográfico (con caché). Reutiliza tu enrich_geo.
    event = enrichment.enrich_geo(event)

    # 4) Señales de secuencia derivadas del historial del usuario:
    #    distancia, velocidad (viaje imposible), cambio de país, dispositivo
    #    nuevo, desajuste horario y ubicación habitual. Se calcula ANTES de
    #    guardar el evento actual, para no compararlo consigo mismo.
    señales_historial = derive_history_signals(event)
    event.update(señales_historial)

    # 5) known_location: la ubicación habitual derivada del historial tiene
    #    prioridad; si el usuario no tiene historial, se usa la que aporte el
    #    cliente y, en su defecto, la ciudad actual (para no penalizar el
    #    primer acceso).
    event["known_location"] = (
        señales_historial.get("known_location")
        or body.get("known_location")
        or event.get("declared_city")
    )

    # 6) Scoring con el motor existente (sin cambios).
    result = score_login_event(event)

    # 7) Persistencia con el módulo existente (sin cambios).
    save_event_result(
        event=event,
        risk_score=result.score,
        risk_level=result.level,
        recommended_action=result.recommended_action,
        risk_reasons=result.reasons,
    )

    return RiskResponse(
        user_id=event.get("user_id"),
        risk_score=result.score,
        risk_level=result.level,
        recommended_action=result.recommended_action,
        risk_reasons=result.reasons,
        message=result.message,
    )

Writing src/pipeline.py


In [18]:
%%writefile src/api_e2e.py
# 8. Creación del nuevo end point

from __future__ import annotations

from typing import Optional

from fastapi import Request
from pydantic import BaseModel, Field

# Importa la app existente y añade una ruta nueva. No modifica
# api_login_analyzer.py: solo registra un endpoint adicional sobre la misma
# instancia.
from src.api_login_analyzer import app
from src.schemas import RiskResponse
from src.pipeline import run_raw_pipeline


class RawLoginBody(BaseModel):
    user_id: str = Field(..., description="Identificador seudonimizado del usuario")
    known_location: Optional[str] = Field(
        default=None,
        description="Ciudad habitual del usuario. Opcional; en el futuro se derivará del historial.",
    )


@app.post("/login-event/raw", response_model=RiskResponse, tags=["end-to-end"])
def analyze_raw_login_event(body: RawLoginBody, request: Request) -> RiskResponse:
    """Analiza un evento de login a partir de la petición real.

    A diferencia de POST /login-event (v4), no exige las señales precalculadas:
    captura IP, user-agent y hora de la propia petición HTTP y deriva la
    geolocalización antes de puntuar.
    """
    return run_raw_pipeline(body.model_dump(), request)


Writing src/api_e2e.py


In [19]:
# 9. Prueba de flujo end-to-end (captura + enriquecimiento + scoring)
import sys, json, os

if os.path.exists("outputs/login_analyzer_events.db"):
    os.remove("outputs/login_analyzer_events.db")
for m in list(sys.modules):
    if m.startswith("src."):
        del sys.modules[m]

from fastapi.testclient import TestClient
import src.api_e2e as api          # importa la app v4 y le añade /login-event/raw
client = TestClient(api.app)

r = client.post("/login-event/raw",
    headers={"User-Agent": "MiNavegador/123", "X-Forwarded-For": "201.244.3.9"},
    json={"user_id": "u_demo", "known_location": "Bogota"})

print("Respuesta del analizador:")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

ev = client.get("/events/recent?limit=1").json()["events"][0]
raw = json.loads(ev["raw_event"])
print("\nEvento guardado:")
print("  ip capturada   :", ev["ip"])
print("  ciudad derivada:", raw.get("declared_city"))
print("  país derivado  :", raw.get("declared_country"))
print("  known_location :", raw.get("known_location"))
print("  ASN            :", raw.get("asn"))

Respuesta del analizador:
{
  "user_id": "u_demo",
  "risk_score": 27,
  "risk_level": "medio",
  "recommended_action": "step_up_mfa",
  "risk_reasons": [
    "localizacion_distinta_a_la_habitual",
    "horario_atipico"
  ],
  "message": "Riesgo medio: se recomienda solicitar verificación adicional antes de permitir el acceso."
}

Evento guardado:
  ip capturada   : 201.244.3.9
  ciudad derivada: Santa Marta
  país derivado  : CO
  known_location : Bogota
  ASN            : AS19429 ETB - Colombia


In [20]:
# 10. Demostracion de la capa de historial: secuencia de viaje imposible
#
# Un mismo usuario inicia sesion primero en Bogota (perfil habitual) y, 20
# minutos despues, aparece en Moscu desde otro dispositivo. El segundo evento
# debe dispararse como riesgo ALTO por acumulacion de senales de secuencia.
import sys, json, os
from datetime import datetime, timezone, timedelta

# Reiniciar la base para partir de un historial limpio
if os.path.exists("outputs/login_analyzer_events.db"):
    os.remove("outputs/login_analyzer_events.db")
for m in list(sys.modules):
    if m.startswith("src."):
        del sys.modules[m]

from fastapi.testclient import TestClient
import src.api_e2e as api
client = TestClient(api.app)

USER = "u_viajero"

# --- Sembrar el perfil habitual: un login en Bogota hace 20 minutos ---
# Se inserta directamente en la BD con una marca temporal anterior, porque la
# captura en vivo siempre usa la hora actual y no permite simular el intervalo.
from src.storage import save_event_result
ts_previo = (datetime.now(timezone.utc) - timedelta(minutes=20)).isoformat()
evento_previo = {
    "user_id": USER, "timestamp": ts_previo, "ip": "201.244.3.9",
    "declared_city": "Bogota", "declared_region": "Bogota D.C.",
    "declared_country": "CO", "timezone_offset": "America/Bogota",
    "asn": "AS3816 ETB", "user_agent": "Chrome/Windows",
    "_lat": 4.61, "_lon": -74.08, "network_type": "residential",
}
save_event_result(event=evento_previo, risk_score=5, risk_level="bajo",
                  recommended_action="allow", risk_reasons=[])
print(f"[Evento 1] Login habitual de {USER} en Bogota (hace 20 min) -> perfil establecido\n")

# --- Cachear la geolocalizacion de la IP de Moscu (reproducibilidad) ---
cache = {}
if os.path.exists("data/geo_cache.json"):
    cache = json.load(open("data/geo_cache.json"))
cache["45.155.205.16"] = {
    "declared_city": "Moscow", "declared_region": "Moscow",
    "declared_country": "RU", "lat": 55.75, "lon": 37.62,
    "timezone_offset": "Europe/Moscow", "asn": "AS9009 M247", "geo_ok": True,
}
os.makedirs("data", exist_ok=True)
json.dump(cache, open("data/geo_cache.json", "w"), ensure_ascii=False, indent=2)

# --- Evento 2: el mismo usuario aparece en Moscu, desde otro dispositivo ---
r = client.post("/login-event/raw",
    headers={"User-Agent": "Firefox/Linux", "X-Forwarded-For": "45.155.205.16"},
    json={"user_id": USER})
j = r.json()

print("[Evento 2] El mismo usuario aparece en Moscu 20 min despues")
print(f"  Nivel de riesgo : {j['risk_level'].upper()}")
print(f"  Score           : {j['risk_score']}")
print(f"  Accion          : {j['recommended_action']}")
print(f"  Razones         : {', '.join(j['risk_reasons'])}\n")

# --- Senales de secuencia derivadas por history.py ---
ev = client.get("/events/recent?limit=1").json()["events"][0]
raw = json.loads(ev["raw_event"])
print("  Senales derivadas del historial:")
print(f"    known_location (habitual) : {raw.get('known_location')}")
print(f"    ciudad del evento         : {raw.get('declared_city')}")
print(f"    distancia recorrida (km)  : {raw.get('distance_from_prev_km')}")
print(f"    tiempo transcurrido (min) : {raw.get('time_since_prev_min')}")
print(f"    velocidad estimada (km/h) : {raw.get('travel_speed_kmh')}")
print(f"    cambio de pais            : {raw.get('country_change')}")
print(f"    dispositivo nuevo         : {raw.get('is_new_device')}")
print(f"    desajuste de zona horaria : {raw.get('timezone_mismatch')}")

[Evento 1] Login habitual de u_viajero en Bogota (hace 20 min) -> perfil establecido

[Evento 2] El mismo usuario aparece en Moscu 20 min despues
  Nivel de riesgo : ALTO
  Score           : 75
  Accion          : block_or_review
  Razones         : localizacion_distinta_a_la_habitual, cambio_de_pais, nuevo_dispositivo, desajuste_de_zona_horaria, horario_atipico, viaje_imposible

  Senales derivadas del historial:
    known_location (habitual) : Bogota
    ciudad del evento         : Moscow
    distancia recorrida (km)  : 10908.77
    tiempo transcurrido (min) : 20.0
    velocidad estimada (km/h) : 32725.55
    cambio de pais            : 1
    dispositivo nuevo         : 1
    desajuste de zona horaria : 1


In [23]:
# Verificación de creación de módulos v5

import os

archivos_v5 = [
    "src/capture.py",
    "src/enrichment.py",
    "src/geo_cache.py",
    "src/history.py",
    "src/pipeline.py",
    "src/api_e2e.py"
]

for archivo in archivos_v5:
    print(archivo, "EXISTE" if os.path.exists(archivo) else "NO EXISTE")

src/capture.py EXISTE
src/enrichment.py EXISTE
src/geo_cache.py EXISTE
src/history.py EXISTE
src/pipeline.py EXISTE
src/api_e2e.py EXISTE


In [24]:
# Celda opcional para descarga local de módulos.
# No es necesaria para la ejecución del prototipo si los archivos ya están en GitHub.

# from google.colab import files
# !zip -j modulos_v5_src.zip \
# src/capture.py \
# src/enrichment.py \
# src/geo_cache.py \
# src/history.py \
# src/pipeline.py \
# src/api_e2e.py
# files.download("modulos_v5_src.zip")